# Image compression with the SVD

*maths · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/svd_compression)

The singular value decomposition writes an image as a sum of rank-1 layers $\sigma_r\,\mathbf u_r\mathbf v_r^T$, strongest first. Keeping only $k$ layers is the best rank-$k$ approximation, its error is exactly the energy of the dropped singular values, and a colour image is three matrices decomposed in one batched call.

**Vector form**

$$A = U\Sigma V^T = \sum_r \sigma_r\,\mathbf u_r \mathbf v_r^T,\qquad A_k = \sum_{r\le k} \sigma_r\,\mathbf u_r \mathbf v_r^T,\qquad \lVert A - A_k\rVert_F^2 = \sum_{r>k}\sigma_r^2$$

**Index (tensor) form**

$$A_{cij} = U_{cir}\,\sigma_{cr}\,V_{cjr},\qquad (A_k)_{cij} = \sum_{r\le k} U_{cir}\,\sigma_{cr}\,V_{cjr},\qquad U_{cir}U_{cis} = \delta_{rs}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{cij}$ | `test_image(size, device)` | (3, H, W) | channel $c$, row $i$, column $j$ of a scene drawn with matplotlib, values in $[0, 1]$ |
| $A_{ij} = w_c A_{cij}$ | `einsum('c,cij->ij', w, rgb)` | (H, W) | repeated $c$ ⇒ summed: brightness with weights $(0.299, 0.587, 0.114)$ |
| $U_{cir},\ \sigma_{cr},\ V_{cjr}$ | `torch.linalg.svd(A, full_matrices=False)` | (3, H, R), (3, R), (3, W, R) | one SVD per channel, batched over the leading index $c$; $R = \min(H, W)$ |
| $(A_k)_{cij}$ | `einsum('...ir,...r,...jr->...ij', U[..., :k], s[..., :k], V[..., :k])` | (3, H, W) | repeated $r \le k$ ⇒ summed: the first $k$ rank-1 layers |
| $\sigma_r U_{ir} V_{jr}$ | `s[r] * einsum('i,j->ij', U[:, r], V[:, r])` | (H, W) | no repeated index: one outer-product layer of the grey image |
| $\sum_{r>k}\sigma_r^2$ | `flip(cumsum(flip(s**2)))` | (R+1,) | Eckart–Young error for every $k$ at once, without rebuilding any image |
| $\mathrm{PSNR}$ | `10 * log10(1 / mse)` | (R,) | peak signal-to-noise ratio in dB, with the mean squared error from the dropped $\sigma$s |

Storing $k$ layers of an $H\times W$ image takes $k(H + W + 1)$ numbers instead of $HW$, so the compression ratio is $HW / k(H + W + 1)$. The Eckart–Young theorem says no rank-$k$ matrix is closer to $A$ in the Frobenius norm, and the error is the sum of the squared singular values left out. The terminal checks this against images actually rebuilt for several $k$. Smooth parts of the scene (the sky gradient) are captured by the first few layers, while sharp edges, diagonals and text need many. That is why the singular values fall quickly at first and then decay slowly, and why JPEG uses local cosine transforms instead. The first layer is roughly the average brightness profile; later layers alternate in sign (red and blue) to add detail. The image is drawn procedurally, so nothing is downloaded. Try a larger image or a higher maximum rank.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/svd_compression/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import svd, tail_energy


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    size = 96 if fast_mode() else params["size"]
    rgb = tensors.test_image(size, dev)  # (3, H, W) in [0, 1]
    a = tensors.grey(rgb)  # (H, W)
    h, w = a.shape
    print(f"synthetic test image: {h}×{w} pixels, 3 colour channels")

    u, s, v = svd(a)  # (H, R), (R,), (W, R)
    u_c, s_c, v_c = svd(rgb)  # batched over channels: (3, H, R), (3, R), (3, W, R)
    print(
        f"grey image: rank {int((s > 1e-5 * s[0]).sum())} of {len(s)}, σ_1..5 = " + ", ".join(f"{x:.2f}" for x in s[:5])
    )
    print(f"orthonormal check: max |UᵀU − I| = {(u.mT @ u - torch.eye(u.shape[1], device=dev)).abs().max().item():.1e}")

    ks = [k for k in (1, 5, 20, 50, 100) if k < len(s)]
    direct = solve.direct_errors(a, u, s, v, ks)
    tail = tail_energy(s)
    total = tail[0].item()
    print("Eckart–Young:  k   ‖A − A_k‖_F²   Σ_{r>k} σ_r²   energy kept")
    for k, d in zip(ks, direct.tolist()):
        print(f"           {k:5d}   {d:12.5f}   {tail[k].item():12.5f}   {1 - tail[k].item() / total:9.3%}")

    rank_ks, psnr, ratio = solve.quality_curve(s_c, h, w)
    for target in (25, 30, 40):
        hit = (psnr >= target).nonzero()
        if len(hit):
            k = int(rank_ks[hit[0, 0]])
            print(f"colour PSNR ≥ {target} dB first at k = {k:3d}: compression ratio {ratio[k - 1].item():5.1f}×")

    k_max = min(params["k_max"], len(s))
    frames = solve.frame_ranks(k_max, params["frames"])
    return plot.animate(
        rgb, (u, s, v), (u_c, s_c, v_c), (rank_ks, psnr, ratio), frames, out_dir / "svd_compression.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/svd_compression/equation.py
r"""Singular value decomposition → tensor translation.

An image is a matrix A_{ij} (row i, column j). Its SVD writes it as a sum of rank-1 layers,

    A_{ij} = \sum_r U_{ir}\,\sigma_r\,V_{jr},       U, V orthonormal columns, \sigma_1 \ge \sigma_2 \ge \dots \ge 0,

and keeping only the first k layers gives the best rank-k approximation (Eckart–Young):

    (A_k)_{ij} = \sum_{r \le k} U_{ir}\,\sigma_r\,V_{jr},     \|A - A_k\|_F^2 = \sum_{r > k} \sigma_r^2.

A colour image A_{cij} is three matrices, one per channel c: torch.linalg.svd batches over the leading index.
"""

import torch


def svd(a: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    r"""A = U Σ Vᵀ for one matrix (H, W) or a batch (C, H, W).   Returns U (..., H, R), σ (..., R), V (..., W, R)."""
    u, s, vh = torch.linalg.svd(a, full_matrices=False)
    return u, s, vh.mT


def reconstruct(u: torch.Tensor, s: torch.Tensor, v: torch.Tensor, k: int) -> torch.Tensor:
    r"""(A_k)_{...ij} = U_{...ir} σ_{...r} V_{...jr}, summed over r ≤ k.   → (..., H, W)"""
    return torch.einsum("...ir,...r,...jr->...ij", u[..., :k], s[..., :k], v[..., :k])


def layer(u: torch.Tensor, s: torch.Tensor, v: torch.Tensor, r: int) -> torch.Tensor:
    r"""One rank-1 layer σ_r U_{ir} V_{jr} (an outer product, nothing summed).   (H, R), (R,), (W, R) → (H, W)"""
    return s[r] * torch.einsum("i,j->ij", u[:, r], v[:, r])


def tail_energy(s: torch.Tensor) -> torch.Tensor:
    r"""Eckart–Young error E_k = Σ_{r>k} σ_r² for k = 0..R: a reversed cumulative sum.   (..., R) → (..., R+1)"""
    tail = torch.flip(torch.cumsum(torch.flip(s**2, [-1]), -1), [-1])  # tail[k] = Σ_{r ≥ k} σ_r²  (0-based)
    return torch.cat([tail, tail.new_zeros(*tail.shape[:-1], 1)], dim=-1)


def psnr(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    r"""Peak signal-to-noise ratio 10 log₁₀(1 / MSE) in dB, for images in [0, 1]."""
    return 10 * torch.log10(1.0 / ((a - b) ** 2).mean())


def compression_ratio(h: int, w: int, k: int) -> float:
    """Numbers stored for the full image divided by those for k layers (k columns of U and V plus k σ's)."""
    return h * w / (k * (h + w + 1))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/svd_compression/tensors.py
"""Build the concrete tensors: a procedurally drawn colour test image (nothing is downloaded) and its grey version."""

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import torch  # noqa: E402
from matplotlib.patches import Circle, Polygon, Rectangle  # noqa: E402

LUMA = [0.299, 0.587, 0.114]  # weights w_c of red, green, blue in the brightness


def test_image(size: int, device: torch.device) -> torch.Tensor:
    """A synthetic scene drawn with matplotlib: sky gradient, sun, hills, a house, stripes and text.

    Returns A_{cij} in [0, 1] with shape (3, size, size): channel c (red, green, blue), row i, column j.
    """
    fig = plt.figure(figsize=(1, 1), dpi=size)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    t = np.linspace(0, 1, 64)[:, None]
    sky = np.stack([0.55 + 0.4 * t, 0.75 + 0.2 * t, np.ones_like(t)], axis=-1)  # (64, 1, 3), light at the bottom
    ax.imshow(sky[::-1], extent=[0, 1, 0.35, 1], aspect="auto")
    ax.add_patch(Circle((0.78, 0.8), 0.09, color="#f59e0b"))
    ax.add_patch(Polygon([[0, 0.35], [0.3, 0.62], [0.62, 0.35]], color="#16a34a"))
    ax.add_patch(Polygon([[0.35, 0.35], [0.72, 0.55], [1, 0.35]], color="#65a30d"))
    ax.add_patch(Rectangle((0, 0), 1, 0.35, color="#a16207"))
    for k in range(6):  # a striped field: sharp edges need many singular values
        ax.add_patch(Rectangle((0, 0.05 * k), 1, 0.025, color="#ca8a04"))
    ax.add_patch(Rectangle((0.1, 0.22), 0.22, 0.2, color="#dc2626"))  # house
    ax.add_patch(Polygon([[0.07, 0.42], [0.21, 0.52], [0.35, 0.42]], color="#7f1d1d"))
    ax.add_patch(Rectangle((0.18, 0.22), 0.06, 0.1, color="#1e3a8a"))
    # The figure is 1 inch = 72 points tall, so font sizes are fractions of the image height.
    ax.text(0.5, 0.1, "A = UΣVᵀ", fontsize=10, ha="center", va="center", color="white", weight="bold")
    ax.text(0.05, 0.86, "SVD", fontsize=13, va="center", color="#1e40af", weight="bold", style="italic")
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())[:size, :size, :3]
    plt.close(fig)
    return torch.tensor(rgba / 255.0, dtype=torch.float32, device=device).permute(2, 0, 1).contiguous()


def grey(rgb: torch.Tensor) -> torch.Tensor:
    """Brightness A_{ij} = w_c A_{cij} (c summed): (3, H, W) → (H, W)."""
    w = torch.tensor(LUMA, dtype=rgb.dtype, device=rgb.device)
    return torch.einsum("c,cij->ij", w, rgb)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/svd_compression/solve.py
"""Decompose the images, check Eckart–Young, and tabulate quality against storage for every rank k."""

import torch

from problems._common.progress import steps

from .equation import compression_ratio, reconstruct, tail_energy


def direct_errors(a: torch.Tensor, u: torch.Tensor, s: torch.Tensor, v: torch.Tensor, ks: list[int]) -> torch.Tensor:
    """‖A − A_k‖_F² by actually rebuilding A_k for each k in ks (grey image (H, W)): (len(ks),)."""
    out = torch.empty(len(ks), dtype=a.dtype, device=a.device)
    for n in steps(len(ks), "ranks"):
        out[n] = ((a - reconstruct(u, s, v, ks[n])) ** 2).sum()
    return out


def quality_curve(s_rgb: torch.Tensor, h: int, w: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """For the colour image, every rank k = 1..R at once, from the singular values alone (Eckart–Young).

    s_rgb: (3, R). Returns ks (R,), PSNR in dB (R,) and the compression ratio (R,).
    """
    r = s_rgb.shape[-1]
    err = tail_energy(s_rgb).sum(dim=0)[1:]  # (R,): Σ_c Σ_{r>k} σ_{cr}², k = 1..R
    mse = err / (3 * h * w)
    psnr = 10 * torch.log10(1.0 / mse.clamp(min=1e-12))
    ks = torch.arange(1, r + 1)
    ratio = torch.tensor([compression_ratio(h, w, int(k)) for k in ks])
    return ks, psnr, ratio


def frame_ranks(k_max: int, n_frames: int) -> list[int]:
    """Ranks shown in the animation: roughly geometric from 1 to k_max, so the early layers get more frames."""
    ks = torch.logspace(0, torch.log10(torch.tensor(float(k_max))), n_frames).round().int().unique()
    return ks.tolist()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/svd_compression/plot.py
"""Animate the rank-k reconstruction growing layer by layer, with the spectrum, the first layers and PSNR vs storage."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import layer, reconstruct


def _image_axis(ax, title: str):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(title, fontsize=9)


def animate(rgb: torch.Tensor, grey_svd: tuple, rgb_svd: tuple, curve: tuple, ranks: list[int], path: Path) -> Path:
    """rgb (3, H, W); grey_svd = (U, σ, V) of the grey image; rgb_svd the batched version; curve = (k, PSNR, ratio)."""
    u, s, v = grey_svd
    u_c, s_c, v_c = rgb_svd
    a = rgb.permute(1, 2, 0).cpu().numpy()  # (H, W, 3) for imshow
    grey = reconstruct(u, s, v, len(s))
    _, psnr, ratio = (t.cpu().numpy() for t in curve)
    s_np, s_c_np = s.cpu().numpy(), s_c.cpu().numpy()

    fig = plt.figure(figsize=(11, 5))
    gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 0.9, 0.9])
    ax_a, ax_k, ax_e = (fig.add_subplot(gs[0, j]) for j in range(3))
    ax_s = fig.add_subplot(gs[0, 3:])
    ax_l = [fig.add_subplot(gs[1, j]) for j in range(3)]
    ax_q = fig.add_subplot(gs[1, 3:])

    ax_a.imshow(a, interpolation="nearest")
    _image_axis(ax_a, "original $A_{cij}$")
    img_k = ax_k.imshow(a, interpolation="nearest")
    _image_axis(ax_k, "")
    img_e = ax_e.imshow(np.zeros(a.shape[:2]), cmap="Greys", vmin=0, vmax=0.5, interpolation="nearest")
    _image_axis(ax_e, "grey error $|A - A_k|$")

    for r, ax in enumerate(ax_l):  # the first rank-1 layers σ_r u_r v_rᵀ of the grey image
        lay = layer(u, s, v, r).cpu().numpy()
        lim = np.abs(lay).max()
        ax.imshow(lay, cmap="RdBu_r", vmin=-lim, vmax=lim, interpolation="nearest")
        _image_axis(ax, rf"layer {r + 1}: $\sigma_{r + 1} u_{r + 1} v_{r + 1}^T$, $\sigma$ = {s_np[r]:.1f}")

    rr = np.arange(1, len(s_np) + 1)
    for c, col in enumerate(["#dc2626", "#16a34a", "#2563eb"]):
        ax_s.semilogy(rr, s_c_np[c], color=col, lw=0.8, alpha=0.7)
    ax_s.semilogy(rr, s_np, color="#111827", lw=1.4, label="grey")
    tail = ax_s.axvspan(1, 1, color="#f59e0b", alpha=0.15, label=r"dropped: $\sum_{r>k}\sigma_r^2$")
    ax_s.set_xlabel("index $r$")
    ax_s.set_ylabel(r"$\sigma_r$")
    ax_s.set_title("singular values (R, G, B and grey)", fontsize=9)
    ax_s.legend(fontsize=7, loc="upper right")

    ax_q.plot(ratio, psnr, color="#2563eb", lw=1.4)
    ax_q.set_xscale("log")
    ax_q.set_xlabel("compression ratio")
    ax_q.set_ylabel("PSNR (dB)")
    ax_q.set_ylim(10, min(60, psnr.max() + 3))
    ax_q.set_title("quality vs storage (colour)", fontsize=9)
    (dot,) = ax_q.plot([], [], "o", color="#dc2626", ms=7)

    def update(n: int):
        k = ranks[n]
        a_k = reconstruct(u_c, s_c, v_c, k).clamp(0, 1)  # all three channels in one batched einsum
        img_k.set_data(a_k.permute(1, 2, 0).cpu().numpy())
        ax_k.set_title(f"rank $k$ = {k}: {ratio[k - 1]:.1f}× smaller", fontsize=9)
        img_e.set_data((grey - reconstruct(u, s, v, k)).abs().cpu().numpy())
        tail.set_x(k + 0.5)
        tail.set_width(len(s_np) - k)
        dot.set_data([ratio[k - 1]], [psnr[k - 1]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(ranks), 120), path, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'size': 256,
    'k_max': 80,
    'frames': 40
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.svd_compression")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.svd_compression.main")
problem_dir = Path("problems/maths/svd_compression")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
